# ENTSO-E Capacity Debug

Dieses Notebook prüft, warum `wind_onshore_capacity_entsoe`, `wind_offshore_capacity_entsoe`, `solar_capacity_entsoe` leer sind.

In [ ]:
import os
from pathlib import Path

import pandas as pd
import polars as pl
from entsoe import EntsoePandasClient

try:
    from dotenv import load_dotenv
except Exception:
    load_dotenv = None

if load_dotenv is not None:
    load_dotenv(Path('.env'))

api_key = os.getenv('ENTSOE_API_KEY')
if not api_key:
    raise RuntimeError('Missing ENTSOE_API_KEY')

REGION_CODE = '10Y1001A1001A83F'  # DE-LU
client = EntsoePandasClient(api_key=api_key)

start = pd.Timestamp('2024-01-01T00:00:00Z')
end = pd.Timestamp('2024-03-01T00:00:00Z')

start, end

(Timestamp('2024-01-01 00:00:00+0000', tz='UTC'),
 Timestamp('2024-03-01 00:00:00+0000', tz='UTC'))

In [11]:
# Helper: normalize index and print structure

def _ensure_utc_index(df):
    if df is None or len(df) == 0:
        return df
    if df.index.tz is None:
        df.index = df.index.tz_localize('UTC')
    else:
        df.index = df.index.tz_convert('UTC')
    return df

def inspect_df(name, df):
    print(f'\n=== {name} ===')
    if df is None:
        print('None')
        return
    print('type:', type(df))
    print('len :', len(df))
    if hasattr(df, 'columns'):
        print('columns type:', type(df.columns))
        print('columns:', list(df.columns))
    print(df.head())


In [12]:
# 1) Installed capacity without psr_type
cap_all = client.query_installed_generation_capacity(REGION_CODE, start=start, end=end, psr_type=None)
cap_all = _ensure_utc_index(cap_all)
inspect_df('capacity_all', cap_all)


=== capacity_all ===
type: <class 'pandas.DataFrame'>
len : 0
columns type: <class 'pandas.Index'>
columns: ['Biomass', 'Fossil Brown coal/Lignite', 'Fossil Coal-derived gas', 'Fossil Gas', 'Fossil Hard coal', 'Fossil Oil', 'Fossil Oil shale', 'Fossil Peat', 'Geothermal', 'Hydro Pumped Storage', 'Hydro Run-of-river and poundage', 'Hydro Water Reservoir', 'Marine', 'Other', 'Other renewable', 'Solar', 'Waste', 'Wind Offshore', 'Wind Onshore']
Empty DataFrame
Columns: [Biomass, Fossil Brown coal/Lignite, Fossil Coal-derived gas, Fossil Gas, Fossil Hard coal, Fossil Oil, Fossil Oil shale, Fossil Peat, Geothermal, Hydro Pumped Storage, Hydro Run-of-river and poundage, Hydro Water Reservoir, Marine, Other, Other renewable, Solar, Waste, Wind Offshore, Wind Onshore]
Index: []


In [13]:
# 2) Installed capacity per PSR type
# Common PSR codes:
# B16=Solar, B18=Wind Offshore, B19=Wind Onshore
psr_map = {
    'solar': 'B16',
    'wind_offshore': 'B18',
    'wind_onshore': 'B19',
}

cap_psr = {}
for name, psr in psr_map.items():
    try:
        df = client.query_installed_generation_capacity(REGION_CODE, start=start, end=end, psr_type=psr)
        df = _ensure_utc_index(df)
        cap_psr[name] = df
        inspect_df(f'capacity_{name}_{psr}', df)
    except Exception as exc:
        print(f'capacity {name} ({psr}) failed: {exc}')


=== capacity_solar_B16 ===
type: <class 'pandas.DataFrame'>
len : 0
columns type: <class 'pandas.Index'>
columns: ['Solar']
Empty DataFrame
Columns: [Solar]
Index: []

=== capacity_wind_offshore_B18 ===
type: <class 'pandas.DataFrame'>
len : 0
columns type: <class 'pandas.Index'>
columns: ['Wind Offshore']
Empty DataFrame
Columns: [Wind Offshore]
Index: []

=== capacity_wind_onshore_B19 ===
type: <class 'pandas.DataFrame'>
len : 0
columns type: <class 'pandas.Index'>
columns: ['Wind Onshore']
Empty DataFrame
Columns: [Wind Onshore]
Index: []


In [14]:
# 3) Wind/Solar generation + forecast for comparison in same window
actuals = client.query_generation(REGION_CODE, start=start, end=end)
actuals = _ensure_utc_index(actuals)
inspect_df('generation_actuals', actuals)

fc_da = client.query_wind_and_solar_forecast(REGION_CODE, start=start, end=end, process_type='A01')
fc_da = _ensure_utc_index(fc_da)
inspect_df('forecast_da', fc_da)

try:
    fc_id = client.query_wind_and_solar_forecast(REGION_CODE, start=start, end=end, process_type='A18')
    fc_id = _ensure_utc_index(fc_id)
    inspect_df('forecast_id_A18', fc_id)
except Exception as exc:
    print('forecast_id_A18 failed:', exc)


=== generation_actuals ===
type: <class 'pandas.DataFrame'>
len : 5760
columns type: <class 'pandas.MultiIndex'>
columns: [('Biomass', 'Actual Aggregated'), ('Fossil Brown coal/Lignite', 'Actual Aggregated'), ('Fossil Coal-derived gas', 'Actual Aggregated'), ('Fossil Gas', 'Actual Aggregated'), ('Fossil Hard coal', 'Actual Aggregated'), ('Fossil Oil', 'Actual Aggregated'), ('Geothermal', 'Actual Aggregated'), ('Hydro Pumped Storage', 'Actual Aggregated'), ('Hydro Pumped Storage', 'Actual Consumption'), ('Hydro Run-of-river and poundage', 'Actual Aggregated'), ('Hydro Water Reservoir', 'Actual Aggregated'), ('Other', 'Actual Aggregated'), ('Other renewable', 'Actual Aggregated'), ('Solar', 'Actual Aggregated'), ('Solar', 'Actual Consumption'), ('Waste', 'Actual Aggregated'), ('Wind Offshore', 'Actual Aggregated'), ('Wind Onshore', 'Actual Aggregated'), ('Wind Onshore', 'Actual Consumption')]
                                    Biomass Fossil Brown coal/Lignite  \
                      

In [15]:
# 4) Build canonical capacity columns (robust to MultiIndex)
def select_capacity_wind_solar(df):
    if df is None or df.empty:
        return None
    if isinstance(df.columns, pd.MultiIndex):
        keep = [c for c in df.columns if c[0] in ['Wind Onshore', 'Wind Offshore', 'Solar']]
        out = df[keep].copy()
        ren = {}
        for c in out.columns:
            if c[0] == 'Wind Onshore':
                ren[c] = 'wind_onshore_capacity_entsoe'
            elif c[0] == 'Wind Offshore':
                ren[c] = 'wind_offshore_capacity_entsoe'
            elif c[0] == 'Solar':
                ren[c] = 'solar_capacity_entsoe'
        return out.rename(columns=ren)
    out = df.copy()
    ren = {
        'Wind Onshore': 'wind_onshore_capacity_entsoe',
        'Wind Offshore': 'wind_offshore_capacity_entsoe',
        'Solar': 'solar_capacity_entsoe',
    }
    keep = [c for c in ren if c in out.columns]
    if not keep:
        return None
    return out[keep].rename(columns={c: ren[c] for c in keep})

cap_norm = select_capacity_wind_solar(cap_all)
if cap_norm is None:
    print('No wind/solar capacity columns found in cap_all')
else:
    idx = pd.date_range(start=start, end=end, freq='1h', tz='UTC', inclusive='left')
    cap_hourly = cap_norm.sort_index().reindex(idx).ffill().bfill()
    print(cap_hourly[['wind_onshore_capacity_entsoe','wind_offshore_capacity_entsoe','solar_capacity_entsoe']].isna().sum())
    cap_hourly.head()

No wind/solar capacity columns found in cap_all


In [16]:
# 5) Optional quick check against raw entsoe.parquet
raw_path = Path('../data/raw/entsoe.parquet')
if raw_path.exists():
    raw = pl.read_parquet(raw_path)
    cap_cols = [c for c in raw.columns if 'capacity_entsoe' in c]
    print('raw capacity cols:', cap_cols)
    if cap_cols:
        print(raw.select([pl.col(c).null_count().alias(c) for c in cap_cols]))
else:
    print('No data/raw/entsoe.parquet found')

raw capacity cols: ['wind_onshore_capacity_entsoe', 'wind_offshore_capacity_entsoe', 'solar_capacity_entsoe']
shape: (1, 3)
┌──────────────────────────────┬───────────────────────────────┬───────────────────────┐
│ wind_onshore_capacity_entsoe ┆ wind_offshore_capacity_entsoe ┆ solar_capacity_entsoe │
│ ---                          ┆ ---                           ┆ ---                   │
│ u64                          ┆ u64                           ┆ u64                   │
╞══════════════════════════════╪═══════════════════════════════╪═══════════════════════╡
│ 44570                        ┆ 44570                         ┆ 44570                 │
└──────────────────────────────┴───────────────────────────────┴───────────────────────┘
